<a href="https://colab.research.google.com/github/Woiny27/NVIDIA-Nemotron-Model-Reasoning-Challenge/blob/main/Snippets_Importing_libraries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing a library that is not in Colaboratory

To import a library that's not in Colaboratory by default, you can use `!pip install` or `!apt-get install`.

## Integrate Google Gemini API for `batch_generate`

To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.

In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GOOGLE_API_KEY`. Then pass the key to the SDK:

In [12]:
# Install the Google Generative AI SDK
!pip install -q -U google-generativeai

import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

# Configure the Gemini API with your API key
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

Now, let's initialize the Generative Model and update the `batch_generate` function to use it. I'll use `gemini-pro` as an example model.

In [13]:
# Initialize the Gemini API model
gemini_model = genai.GenerativeModel('gemini-pro')

# Updated batch_generate function using the Gemini API
def batch_generate(prompts, batch_size):
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        for prompt_text in batch:
            # Construct a system instruction to guide the model's output format
            system_instruction = "You are a reasoning assistant. Solve the problem carefully and give a concise answer in the format: 'Reasoning: [your reasoning] Answer: [your answer]'."
            try:
                response = gemini_model.generate_content(f'{system_instruction}\n\nProblem: {prompt_text}')
                results.append(response.text)
            except Exception as e:
                # Basic error handling: if generation fails, return a default string
                print(f"Error generating content for prompt: {prompt_text} - {e}")
                results.append(f"Reasoning: Generation failed due to an error. Answer: Error for '{prompt_text}'")
    return results

print("batch_generate function updated to use Gemini API.")

batch_generate function updated to use Gemini API.


In [2]:
!pip install matplotlib-venn

In [3]:
!apt-get -qq install -y libfluidsynth1

E: Package 'libfluidsynth1' has no installation candidate


In [10]:
import pandas as pd
import numpy as np
import torch

# ==========================================
# Add this line to load your test data.
# Replace 'test.csv' with the actual path to your test dataset.
# For example: test = pd.read_csv('/kaggle/input/your-dataset-name/test.csv')
try:
    test = pd.read_csv('test.csv') # Placeholder: You might need to adjust the file path
except FileNotFoundError:
    print("Error: 'test.csv' not found. Please ensure the test dataset is available and path is correct.")
    # Or, if 'test' should be defined differently, add its definition here.
    # For demonstration, creating a dummy DataFrame if the file isn't found.
    test = pd.DataFrame({"id": [1, 2, 3], "question": ["What is 1+1?", "What is 2+2?", "What is 3+3?"]})


# ==========================================
# Define the batch_generate function (Placeholder - Replace with your actual model's generation logic)
def batch_generate(prompts, batch_size):
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        for prompt in batch:
            # Simulate a simple generation based on the prompt
            if "1+1" in prompt:
                results.append("Reasoning: This is a simple arithmetic operation. Answer: 2")
            elif "2+2" in prompt:
                results.append("Reasoning: Basic addition. Answer: 4")
            elif "3+3" in prompt:
                results.append("Reasoning: Another basic sum. Answer: 6")
            else:
                results.append(f"Reasoning: I'm not sure, but here's a guess. Answer: Dummy output for '{prompt}'")
    return results


# ==========================================
# 1. RUN BATCH GENERATION FOR REASONING & ANSWER
# ==========================================
# Assuming your generation loop is running, we extract the full text
question_col = [c for c in test.columns if "question" in c.lower() or "prompt" in c.lower()][0]
raw_outputs = batch_generate(test[question_col].tolist(), batch_size=4)

reasonings = []
answers = []

for output in raw_outputs:
    # Split the output to separate the model's inner reasoning from its final answer
    if "Answer:" in output:
        parts = output.split("Answer:")
        # Everything before "Answer:" is the setup/reasoning process
        reasoning_text = parts[0].replace("You are a reasoning assistant. Solve the problem carefully and give a concise answer.", "").strip()
        answer_text = parts[1].strip()
    else:
        reasoning_text = "Analysis completed."
        answer_text = output.strip()

    reasonings.append(reasoning_text)
    answers.append(answer_text)

# ==========================================
# 2. ORGANIZE INTO THE EXACT REQUIRED FORMAT
# ==========================================
submission = pd.DataFrame({
    "id": test["id"],
    "reasoning": reasonings,
    "answer": answers
})

# Save to your local directory
submission.to_csv("submission.csv", index=False)

# ==========================================
# 3. INTEGRITY & FORMAT VALIDATION
# ==========================================
print("---- Final Submission Sample Check ----")
print(submission.head(3))
print(f"\nTotal Rows: {submission.shape[0]}")

assert list(submission.columns) == ["id", "reasoning", "answer"], "Column names do not match template!"
print("\n[SUCCESS] 'submission.csv' matches your exact 3-column format layout!")

Error: 'test.csv' not found. Please ensure the test dataset is available and path is correct.
---- Final Submission Sample Check ----
   id                                          reasoning answer
0   1  Reasoning: This is a simple arithmetic operation.      2
1   2                         Reasoning: Basic addition.      4
2   3                      Reasoning: Another basic sum.      6

Total Rows: 3

[SUCCESS] 'submission.csv' matches your exact 3-column format layout!


In [15]:
# Load the 'test' DataFrame from a CSV file
# Replace 'path/to/your/test.csv' with the actual path to your CSV file
try:
    test = pd.read_csv('path/to/your/test.csv')
    print("Test DataFrame loaded successfully from CSV.")
    display(test.head())
except FileNotFoundError:
    print("Error: The specified 'test.csv' file was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading the test DataFrame: {e}")

Error: The specified 'test.csv' file was not found. Please check the path.


In [16]:
import pandas as pd

# Load the 'test' DataFrame from 'test.csv'
# Make sure 'test.csv' is uploaded to your Colab environment (e.g., in the files sidebar)
# or specify the correct path to the file (e.g., '/content/your_folder/test.csv')
try:
    test = pd.read_csv('test.csv')
    print("Test DataFrame loaded successfully from 'test.csv'.")
    display(test.head())
except FileNotFoundError:
    print("Error: 'test.csv' not found. Please upload the file or specify the correct path.")
except Exception as e:
    print(f"An error occurred while loading the test DataFrame: {e}")

Error: 'test.csv' not found. Please upload the file or specify the correct path.


In [14]:
print("Iterating over the 'test' DataFrame:")
for index, row in test.iterrows():
    print(f"ID: {row['id']}, Question: {row['question']}")

Iterating over the 'test' DataFrame:
ID: 1, Question: What is 1+1?
ID: 2, Question: What is 2+2?
ID: 3, Question: What is 3+3?


In [11]:
display(submission['reasoning'])

,reasoning
0,Reasoning: This is a simple arithmetic operation.
1,Reasoning: Basic addition.
2,Reasoning: Another basic sum.


# Install 7zip reader [libarchive](https://pypi.python.org/pypi/libarchive)

In [4]:
# https://pypi.python.org/pypi/libarchive
!apt-get -qq install -y libarchive-dev && pip install -U libarchive
import libarchive

Selecting previously unselected package libarchive-dev:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libarchive-dev_3.6.0-1ubuntu1.7_amd64.deb ...
Unpacking libarchive-dev:amd64 (3.6.0-1ubuntu1.7) ...
Setting up libarchive-dev:amd64 (3.6.0-1ubuntu1.7) ...
Processing triggers for man-db (2.10.2-1) ...
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 11.3 MB/s eta 0:00:00
  Created wheel for libarchive: filename=libarchive-0.4.7-py3-none-any.whl size=31629 sha256=df545cba70edc055a44cf10a010446e4e1e9eba402b1a8f75840cf8fbd2b973f
  Stored in directory: /root/.cache/pip/wheels/29/20/ab/f101da7b245b996aa097685ef742243725ea6150f5b3b6d9ed
Successfully built libarchive


# Install GraphViz & [PyDot](https://pypi.python.org/pypi/pydot)

In [5]:
# https://pypi.python.org/pypi/pydot
!apt-get -qq install -y graphviz && pip install pydot
import pydot

# Install [cartopy](http://scitools.org.uk/cartopy/docs/latest/)

In [6]:
!pip install cartopy
import cartopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 58.1 MB/s eta 0:00:00
